# SOAL 5 – PENGGABUNGAN DATA PARIWISATA NTB

**Mata Kuliah:** Data Acquisition & Management  
**Topik:** Pariwisata Nusa Tenggara Barat (NTB)

Notebook ini menggabungkan minimal dua sumber data:
1. Dataset kunjungan wisatawan NTB dari Satu Data NTB.
2. Data cuaca dari API Open-Meteo.

Kunci penggabungan adalah **tahun dan bulan**. Penggabungan menggunakan `pandas.merge()` dengan `validate='many_to_one'`.

**Prasyarat:** Jalankan terlebih dahulu bagian akuisisi dataset utama dan API pada notebook utama. Notebook ini mengharapkan DataFrame `df` (dataset wisatawan) dan `df_api` (data cuaca harian) sudah tersedia. Jika belum, jalankan sel upload dan pembacaan data pada bagian berikut.

In [ ]:
import os
import json
import pandas as pd
from pathlib import Path

# Folder proyek. Sesuaikan jika lokasi folder di Colab berbeda.
PROJECT_DIR = Path("/content/UTS-DAM-Pariwisata-NTB")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
METADATA_DIR = PROJECT_DIR / "metadata"

for folder in [RAW_DIR, PROCESSED_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folder proyek siap:", PROJECT_DIR)

## 1. Periksa data masukan

Pastikan `df` berisi dataset Excel wisatawan NTB dan `df_api` berisi hasil API cuaca Open-Meteo. Data API minimal memiliki kolom `date`, `temperature_2m_mean`, `precipitation_sum`, dan `wind_speed_10m_max`.

In [ ]:
# Periksa apakah kedua DataFrame sudah tersedia
try:
    print("Dataset wisatawan:", df.shape)
    print("Dataset API:", df_api.shape)
except NameError as e:
    raise NameError(
        "DataFrame df atau df_api belum tersedia. "
        "Jalankan dahulu sel pembacaan Excel dan akuisisi API di notebook utama."
    ) from e

display(df.head())
display(df_api.head())

## 2. Bersihkan dan siapkan dataset wisatawan

File mentah asli tidak diubah. Pembersihan dilakukan pada salinan DataFrame untuk analisis.

In [ ]:
wisata = df.copy()

# Standarkan nama kolom
wisata.columns = (
    wisata.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Petakan nama kolom dari file sumber ke nama standar
rename_map = {
    "nama_provinsi": "provinsi",
    "kategori_wisatawan": "kategori",
    "jumlah_orang": "jumlah"
}
wisata = wisata.rename(columns=rename_map)

required_wisata = {"tahun", "bulan", "kategori", "jumlah"}
missing = required_wisata - set(wisata.columns)
if missing:
    raise ValueError(f"Kolom dataset wisatawan tidak ditemukan: {sorted(missing)}")

# Konversi tipe data
wisata["tahun"] = pd.to_numeric(wisata["tahun"], errors="coerce")
wisata["jumlah"] = pd.to_numeric(
    wisata["jumlah"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

# Standarkan ejaan kategori pada salinan analisis
wisata["kategori"] = (
    wisata["kategori"].astype(str).str.strip()
    .replace({"Wisatawan Mancaegara": "Wisatawan Mancanegara"})
)

bulan_map = {
    "januari": 1, "februari": 2, "maret": 3, "april": 4,
    "mei": 5, "juni": 6, "juli": 7, "agustus": 8,
    "september": 9, "oktober": 10, "november": 11, "desember": 12
}
wisata["bulan"] = (
    wisata["bulan"].astype(str).str.strip().str.lower().map(bulan_map)
)

# Pastikan kolom kunci dan jumlah valid
wisata = wisata.dropna(subset=["tahun", "bulan", "jumlah"]).copy()
wisata["tahun"] = wisata["tahun"].astype(int)
wisata["bulan"] = wisata["bulan"].astype(int)

# Pilih periode tahun 2024 agar sama dengan periode API
wisata_2024 = wisata.loc[wisata["tahun"] == 2024].copy()

if wisata_2024.empty:
    raise ValueError("Tidak ada data wisatawan untuk tahun 2024.")

print("Jumlah baris wisatawan tahun 2024:", len(wisata_2024))
display(wisata_2024.head())

## 3. Siapkan dan agregasikan data API cuaca

Data cuaca Open-Meteo umumnya harian. Data tersebut diringkas menjadi satu baris per bulan. Suhu dan kecepatan angin dihitung rata-ratanya, sedangkan curah hujan dijumlahkan.

In [ ]:
cuaca = df_api.copy()

required_api = {
    "date", "temperature_2m_mean",
    "precipitation_sum", "wind_speed_10m_max"
}
missing_api = required_api - set(cuaca.columns)
if missing_api:
    raise ValueError(f"Kolom API tidak ditemukan: {sorted(missing_api)}")

cuaca["date"] = pd.to_datetime(cuaca["date"], errors="coerce")
cuaca = cuaca.dropna(subset=["date"]).copy()
cuaca["tahun"] = cuaca["date"].dt.year
cuaca["bulan"] = cuaca["date"].dt.month

cuaca_2024 = cuaca.loc[cuaca["tahun"] == 2024].copy()
if cuaca_2024.empty:
    raise ValueError("Data API tidak berisi data tahun 2024.")

cuaca_bulanan = (
    cuaca_2024
    .groupby(["tahun", "bulan"], as_index=False)
    .agg(
        suhu_rata_rata_c=("temperature_2m_mean", "mean"),
        curah_hujan_mm=("precipitation_sum", "sum"),
        kecepatan_angin_rata_rata_kmh=("wind_speed_10m_max", "mean")
    )
)

# Satu baris cuaca harus mewakili satu kombinasi tahun-bulan
assert not cuaca_bulanan.duplicated(["tahun", "bulan"]).any()

print("Jumlah bulan cuaca:", len(cuaca_bulanan))
display(cuaca_bulanan.head(12))

## 4. Gabungkan data menggunakan merge dan validate

Dipakai `how='left'` agar seluruh catatan kunjungan wisatawan tahun 2024 tetap dipertahankan. `validate='many_to_one'` sesuai karena banyak kategori wisatawan dapat berada pada bulan yang sama, tetapi tabel cuaca hanya memiliki satu catatan per bulan.

In [ ]:
df_gabungan = pd.merge(
    wisata_2024,
    cuaca_bulanan,
    on=["tahun", "bulan"],
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Status pencocokan data:")
print(df_gabungan["_merge"].value_counts())

# Pastikan seluruh baris wisatawan memperoleh pasangan data cuaca
if not (df_gabungan["_merge"] == "both").all():
    unmatched = df_gabungan.loc[
        df_gabungan["_merge"] != "both", ["tahun", "bulan"]
    ].drop_duplicates()
    raise ValueError(
        "Ada bulan yang tidak memiliki pasangan data cuaca:\n"
        + unmatched.to_string(index=False)
    )

# Validasi jumlah baris
assert len(df_gabungan) == len(wisata_2024)

# Periksa duplikasi baris penuh
jumlah_duplikat = int(df_gabungan.duplicated().sum())
print("Baris sebelum merge:", len(wisata_2024))
print("Baris sesudah merge:", len(df_gabungan))
print("Jumlah duplikasi baris:", jumlah_duplikat)

assert jumlah_duplikat == 0, "Ditemukan baris duplikat pada hasil merge!"

df_gabungan = df_gabungan.drop(columns=["_merge"])
display(df_gabungan.head(10))

## 5. Simpan hasil gabungan dan metadata

Hasil gabungan disimpan sebagai CSV. Metadata mencatat sumber, kunci penggabungan, metode, jumlah baris/kolom, dan hasil pemeriksaan duplikasi.

In [ ]:
output_csv = PROCESSED_DIR / "dataset_gabungan_pariwisata_ntb.csv"
df_gabungan.to_csv(output_csv, index=False, encoding="utf-8-sig")

metadata_merge = {
    "nama_proyek": "Analisis Data Pariwisata Nusa Tenggara Barat",
    "nama_output": output_csv.name,
    "sumber_1": "Satu Data NTB - Jumlah Kunjungan Wisatawan berdasarkan Jenis Wisatawan",
    "url_sumber_1": "https://data.ntbprov.go.id/dataset/9e270f88-a604-4640-a4f0-24c996122b92/show",
    "sumber_2": "Open-Meteo Historical Weather API",
    "url_sumber_2": "https://archive-api.open-meteo.com/v1/archive",
    "kunci_penggabungan": ["tahun", "bulan"],
    "metode": "left",
    "validate": "many_to_one",
    "periode": 2024,
    "jumlah_baris": int(len(df_gabungan)),
    "jumlah_kolom": int(len(df_gabungan.columns)),
    "jumlah_duplikasi_baris": int(df_gabungan.duplicated().sum()),
    "status_pemeriksaan": "Lolos" if df_gabungan.duplicated().sum() == 0 else "Perlu pemeriksaan"
}

metadata_path = METADATA_DIR / "metadata_merge.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata_merge, f, indent=4, ensure_ascii=False)

print("CSV hasil gabungan:", output_csv)
print("Metadata:", metadata_path)
print("Ukuran hasil:", df_gabungan.shape)
print("Duplikasi:", df_gabungan.duplicated().sum())

## 6. Kesimpulan Soal 5

Penggabungan dilakukan antara dataset kunjungan wisatawan NTB dan data cuaca Open-Meteo periode 2024. Kolom `tahun` dan `bulan` digunakan sebagai kunci yang bermakna. Penggabungan memakai `left join` dan `validate='many_to_one'` karena beberapa kategori wisatawan dapat memiliki periode bulan yang sama, sedangkan data cuaca diringkas menjadi satu baris per bulan.

Keberhasilan penggabungan dibuktikan melalui indikator pencocokan (`_merge`), kesamaan jumlah baris sebelum dan sesudah penggabungan, serta pemeriksaan duplikasi. Hasil disimpan pada `data/processed/dataset_gabungan_pariwisata_ntb.csv` dan metadata disimpan pada `metadata/metadata_merge.json`.

**Catatan:** Kesimpulan akhir hanya dinyatakan berhasil jika semua sel validasi di atas berjalan tanpa error.